# Evo-1 + LIBERO Setup on Google Colab

This notebook sets up both the Evo-1 server (for inference) and LIBERO environment (for evaluation) on Google Colab with GPU support.

## 1. Check GPU Availability

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ No GPU detected! Go to Runtime > Change runtime type > GPU")

## 2. Install Dependencies

**Note:** Flash Attention installation may fail on some Colab environments. This is OK - the code includes fallback logic and will still work, just slightly slower.

In [ ]:
%%bash
# Install system dependencies
apt-get update -qq
apt-get install -y git wget

In [ ]:
# Install Python packages
print("Installing PyTorch with CUDA support...")
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

print("\nInstalling core model dependencies...")
!pip install -q transformers==4.45.2 timm==1.0.9 accelerate==1.2.1 diffusers==0.31.0

print("\nInstalling WebSocket and utilities...")
!pip install -q websockets pyyaml opencv-python pillow numpy fvcore

print("\nInstalling additional dependencies...")
!pip install -q einops thop cloudpickle easydict hydra-core wandb

print("\nInstalling gym and robotics packages...")
!pip install -q gym==0.25.2

print("\nInstalling Flash Attention (this may take 5-10 minutes)...")
print("Note: If this fails, the code will still work but may be slower")
# Install build dependencies
!pip install -q ninja packaging wheel setuptools

# Try to install flash-attn with proper environment variables
import subprocess
import sys

try:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "flash-attn", "--no-build-isolation"],
        env={**os.environ, "MAX_JOBS": "4", "FLASH_ATTENTION_FORCE_BUILD": "TRUE"},
        capture_output=True,
        text=True,
        timeout=600  # 10 minute timeout
    )
    if result.returncode == 0:
        print("✅ Flash Attention installed successfully!")
    else:
        print("⚠️  Flash Attention installation failed (this is OK - code will use fallback)")
        print("The model will work but may be slightly slower")
except Exception as e:
    print(f"⚠️  Flash Attention installation failed: {e}")
    print("The model will work but may be slightly slower")

print("\n✅ All critical dependencies installed successfully!")

## 3. Clone Evo-1 Repository

In [ ]:
import os
from pathlib import Path

# Clone Evo-1 repository
if not Path("/content/Evo-1").exists():
    !git clone https://github.com/MINT-SJTU/Evo-1.git /content/Evo-1
    print("✅ Repository cloned")
else:
    print("✅ Repository already exists")

os.chdir("/content/Evo-1")

In [ ]:
%%bash
cd /content/Evo-1/Evo_1/
pip install -r requirements.txt

## 4. Download Model Checkpoint

In [ ]:
from huggingface_hub import snapshot_download
import os

# Create checkpoint directory
checkpoint_dir = "/content/Evo-1/checkpoints/libero"
os.makedirs(checkpoint_dir, exist_ok=True)

# Download checkpoint from HuggingFace
print("Downloading Evo-1 LIBERO checkpoint (1.4GB)...")
snapshot_download(
    repo_id="MINT-SJTU/Evo1_LIBERO",
    local_dir=checkpoint_dir,
    local_dir_use_symlinks=False
)

print("✅ Checkpoint downloaded")
print("\nCheckpoint files:")
!ls -lh {checkpoint_dir}

## 5. Configure Model for CUDA

In [ ]:
import json

# Update config to use CUDA
config_path = "/content/Evo-1/checkpoints/libero/config.json"
with open(config_path, 'r') as f:
    config = json.load(f)

config['device'] = 'cuda'

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print("✅ Config updated to use CUDA")
print(f"Device: {config['device']}")

## 6. Fix Server Code for Multi-Device Support

In [ ]:
# Fix Evo1_server.py to support dynamic device selection
server_file = "/content/Evo-1/Evo_1/scripts/Evo1_server.py"

# Read the file
with open(server_file, 'r') as f:
    content = f.read()

# Fix 1: Update load_model_and_normalizer to use device from config
content = content.replace(
    'model = model.to("cuda")',
    'device = config.get("device", "cuda")\n    model = model.to(device)'
)

# Fix 2: Update decode_image_from_list signature
content = content.replace(
    'def decode_image_from_list(img_list):',
    'def decode_image_from_list(img_list, device="cuda"):'
)

# Fix 3: Update decode_image_from_list to use device parameter
content = content.replace(
    'return transforms.ToTensor()(pil).to("cuda")',
    'return transforms.ToTensor()(pil).to(device)'
)

# Fix 4: Update infer_from_json_dict
content = content.replace(
    'def infer_from_json_dict(data: dict, model, normalizer):\n    device = "cuda"',
    'def infer_from_json_dict(data: dict, model, normalizer):\n    device = next(model.parameters()).device'
)

# Fix 5: Update image decoding call
content = content.replace(
    'images = [decode_image_from_list(img) for img in data["image"]]',
    'images = [decode_image_from_list(img, device=device) for img in data["image"]]'
)

# Write back
with open(server_file, 'w') as f:
    f.write(content)

print("✅ Server code fixed for device support")

In [ ]:
# Fix InternVL3 embedder for device-specific configuration
embedder_file = "/content/Evo-1/Evo_1/model/internvl3/internvl3_embedder.py"

with open(embedder_file, 'r') as f:
    content = f.read()

# Replace hardcoded torch_dtype and use_flash_attn with conditional logic
old_code = '''        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=False)
        self.model = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16,
            trust_remote_code=True,
            use_flash_attn=True,
            low_cpu_mem_usage=True,
            _fast_init=False,
        ).to(self.device)'''

new_code = '''        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=False)
        
        # Use appropriate dtype and features based on device
        if device == "cuda":
            torch_dtype = torch.bfloat16
            use_flash_attn = True
        else:
            # MPS/CPU: use float32 and disable flash attention
            torch_dtype = torch.float32
            use_flash_attn = False
        
        self.model = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch_dtype,
            trust_remote_code=True,
            use_flash_attn=use_flash_attn,
            low_cpu_mem_usage=True,
            _fast_init=False,
        ).to(self.device)'''

content = content.replace(old_code, new_code)

with open(embedder_file, 'w') as f:
    f.write(content)

print("✅ Embedder code fixed for device support")

## 7. Install WebSocket Server Dependencies

In [ ]:
# Install websockets for the server (already installed in cell 2 but make sure it's available)
print("📦 Ensuring websockets is installed for server...")
!pip install -q websockets

print("✅ WebSocket dependencies ready")

## 7. Setup Cloudflare Tunnel (For Remote Access)

Cloudflare Tunnel is free, stable, and perfect for long-running sessions. No credit card required!

In [ ]:
# Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

print("✅ Cloudflare Tunnel installed")
print("\nCloudflare Tunnel will start automatically when you run the server cell.")

In [ ]:
# Create a simple WebSocket-aware proxy script
proxy_code = '''
import asyncio
import websockets
from aiohttp import web
import json

# WebSocket server connection
WS_SERVER = "ws://localhost:9000"

async def proxy_handler(request):
    """Handle incoming WebSocket connections and proxy to local server"""
    ws_response = web.WebSocketResponse()
    await ws_response.prepare(request)
    
    print(f"Client connected from {request.remote}")
    
    try:
        # Connect to local WebSocket server
        async with websockets.connect(WS_SERVER) as ws_server:
            print("Connected to local server")
            
            # Bidirectional proxy
            async def forward_to_server():
                async for msg in ws_response:
                    if msg.type == web.WSMsgType.TEXT:
                        await ws_server.send(msg.data)
                    elif msg.type == web.WSMsgType.ERROR:
                        print(f"WebSocket error: {ws_response.exception()}")
                        break
            
            async def forward_to_client():
                async for msg in ws_server:
                    await ws_response.send_str(msg)
            
            # Run both directions concurrently
            await asyncio.gather(
                forward_to_server(),
                forward_to_client(),
                return_exceptions=True
            )
    except Exception as e:
        print(f"Proxy error: {e}")
    finally:
        await ws_response.close()
        print("Client disconnected")
    
    return ws_response

async def start_proxy():
    app = web.Application()
    app.router.add_get("/", proxy_handler)
    
    runner = web.AppRunner(app)
    await runner.setup()
    site = web.TCPSite(runner, "0.0.0.0", 8080)
    await site.start()
    
    print("WebSocket proxy running on port 8080")
    await asyncio.Future()  # Run forever

if __name__ == "__main__":
    asyncio.run(start_proxy())
'''

with open('/content/ws_proxy.py', 'w') as f:
    f.write(proxy_code)

# Install aiohttp for the proxy
!pip install -q aiohttp

print("✅ WebSocket proxy script created")

## 8. Start Evo-1 Server

This will:
1. Load the 1B parameter InternVL3 model (~2-3 minutes)
2. Start WebSocket server on port 9000
3. Create Cloudflare Tunnel for remote access

In [ ]:
import os
import sys
import threading
import time
import subprocess
import re

# Change to scripts directory
os.chdir("/content/Evo-1/Evo_1/scripts")

# Update checkpoint path in server
with open("Evo1_server.py", 'r') as f:
    server_content = f.read()

server_content = server_content.replace(
    'ckpt_dir = "/Users/tmprithvi/Code/EmbodiedLLM/vla-benchmark/models/Evo-1/checkpoints/libero"',
    'ckpt_dir = "/content/Evo-1/checkpoints/libero"'
)

with open("Evo1_server.py", 'w') as f:
    f.write(server_content)

print("🚀 Starting Evo-1 server...")
print("Loading model (this will take 2-3 minutes)...\n")

# Start server in background thread
def run_server():
    os.system("python Evo1_server.py")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for server to start
print("Waiting for server to load model...")
time.sleep(90)  # Give it more time to fully load

print("\n🚀 Starting WebSocket proxy on port 8080...")
# Start proxy in background thread
def run_proxy():
    os.system("python /content/ws_proxy.py")

proxy_thread = threading.Thread(target=run_proxy, daemon=True)
proxy_thread.start()

# Give proxy time to start
time.sleep(3)

# Start Cloudflare Tunnel pointing to proxy (port 8080)
print(f"\n🌐 Starting Cloudflare Tunnel...")
tunnel_process = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8080'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Wait for tunnel URL
public_url = None
for line in tunnel_process.stdout:
    print(line.strip())
    if 'trycloudflare.com' in line or '.cfargotunnel.com' in line:
        # Extract URL from output
        match = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', line)
        if match:
            public_url = match.group(0)
            # Convert https to wss for WebSocket
            public_url = public_url.replace('https://', 'wss://')
            break

if public_url:
    print("\n" + "="*60)
    print("✅ Server is running!")
    print("="*60)
    print(f"\n🌐 Public URL: {public_url}")
    print("\n📝 Update your local client to use this URL:")
    print(f"   SERVER_URL = '{public_url}'")
    print("\n⚠️  Keep this cell running! The server will stop if you interrupt it.")
    print("="*60)
else:
    print("\n⚠️  Could not detect tunnel URL. Check output above.")

# Keep the cell running
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nStopping server and tunnel...")
    tunnel_process.terminate()

## 9. Test Server Connection (Optional)

In [ ]:
import asyncio
import websockets
import json
import numpy as np

async def test_connection():
    # Test data
    test_data = {
        "image": [np.random.randint(0, 255, (480, 640, 3)).tolist() for _ in range(3)],
        "state": np.zeros(24).tolist(),
        "prompt": "test prompt",
        "image_mask": [1, 1, 1],
        "action_mask": 1
    }
    
    try:
        async with websockets.connect("ws://localhost:9000") as ws:
            await ws.send(json.dumps(test_data))
            response = await ws.recv()
            print("✅ Server connection successful!")
            print(f"Response shape: {np.array(json.loads(response)).shape}")
    except Exception as e:
        print(f"❌ Connection failed: {e}")

# Run test using asyncio.run() for proper async context
asyncio.run(test_connection())

## 10. Monitor GPU Usage

In [ ]:
!nvidia-smi

## Next Steps

1. **Copy the Cloudflare URL** from step 8 above (looks like `wss://xxx.trycloudflare.com`)
2. **On your local machine**, update the client script:
   ```python
   SERVER_URL = "wss://xxx.trycloudflare.com"  # Use your Cloudflare URL
   ```
   Edit this file: `/Users/tmprithvi/Code/EmbodiedLLM/vla-benchmark/models/Evo-1/LIBERO_evaluation/libero_client_4tasks.py`
   
3. **Run the LIBERO client** from your Mac:
   ```bash
   cd /Users/tmprithvi/Code/EmbodiedLLM/vla-benchmark
   ./scripts/run_libero_client.sh
   ```

4. **Expected runtime**: 2-3 hours for full evaluation
5. **Expected success rate**: ~94.8% (from paper)

---

**Tips:**
- Keep this notebook running during evaluation
- Monitor GPU usage in step 10
- Cloudflare Tunnel is more stable than ngrok for long sessions
- Colab will disconnect after ~12 hours (even with Pro)
- Save checkpoint if you need to resume

## 11. Setup LIBERO Environment (Optional - for running client on Colab too)

If you want to run both server AND client on Colab instead of your Mac:

In [ ]:
# Install conda on Colab if not already installed
import os
import sys

if not os.path.exists("/usr/local/bin/conda"):
    print("📦 Installing Miniconda...")
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
    !bash Miniconda3-latest-Linux-x86_64.sh -b -p /usr/local -f
    !rm Miniconda3-latest-Linux-x86_64.sh
    print("✅ Miniconda installed")
else:
    print("✅ Conda already installed")

# Initialize conda for bash
!conda init bash
!/bin/bash -c "source /root/.bashrc"

print("\n📦 Creating LIBERO conda environment with Python 3.8.13...")
# Create the conda environment
!conda create -n libero python=3.8.13 -y

print("✅ Conda environment 'libero' created")

In [ ]:
# Clone and setup LIBERO in conda environment
print("📦 Cloning LIBERO repository...")

# Create directory and clone
!mkdir -p /content/LIBERO_evaluation
%cd /content/LIBERO_evaluation

if not os.path.exists("LIBERO"):
    !git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git
    print("✅ LIBERO cloned")
else:
    print("✅ LIBERO already exists")

%cd LIBERO

print("\n📦 Installing LIBERO dependencies in conda environment...")
# Install all dependencies in the libero conda environment
!conda run -n libero pip install -r requirements.txt

print("\n📦 Installing PyTorch with CUDA 11.3...")
!conda run -n libero pip install torch==1.11.0+cu113 torchvision==0.12.0+cu113 torchaudio==0.11.0 --extra-index-url https://download.pytorch.org/whl/cu113

print("\n📦 Installing LIBERO package...")
!conda run -n libero pip install -e .

print("\n📦 Installing WebSocket support...")
!conda run -n libero pip install websockets huggingface_hub

print("\n✅ LIBERO environment setup complete!")
print("Environment: libero (Python 3.8.13)")
print("All dependencies installed in isolated conda environment")

## 12. Run LIBERO Client on Colab

Now you can run the LIBERO evaluation directly on Colab, connecting to the local server via `ws://localhost:9000`:

In [ ]:
# Run LIBERO client in the conda environment
print("📝 To run LIBERO evaluation on Colab:")
print("\n1. Upload your libero_client_4tasks.py to /content/LIBERO_evaluation/")
print("2. Update SERVER_URL in the script to 'ws://localhost:9000'")
print("3. Run the client using conda:")
print("   !conda run -n libero python libero_client_4tasks.py")
print("\n✅ Benefits of using conda:")
print("   - Proper Python 3.8.13 environment (LIBERO requirement)")
print("   - Isolated dependencies (won't conflict with Evo-1)")
print("   - Same environment as your Mac setup")
print("\n⚠️  Or run client on your Mac for easier debugging (see 'Next Steps' in cell 10)")

## 13. Run Both Server and Client on Colab (No Tunnel Needed)

In [ ]:
import os
import threading
import time
import subprocess
import sys

# Kill any existing process on port 9000
print("🧹 Cleaning up any existing server on port 9000...")
!fuser -k 9000/tcp 2>/dev/null || true
time.sleep(2)

# Change to Evo-1 scripts directory
os.chdir("/content/Evo-1/Evo_1/scripts")

# Update checkpoint path in server
with open("Evo1_server.py", 'r') as f:
    server_content = f.read()

server_content = server_content.replace(
    'ckpt_dir = "/Users/tmprithvi/Code/EmbodiedLLM/vla-benchmark/models/Evo-1/checkpoints/libero"',
    'ckpt_dir = "/content/Evo-1/checkpoints/libero"'
)

with open("Evo1_server.py", 'w') as f:
    f.write(server_content)

print("🚀 Starting Evo-1 server...")
print("Loading model (this will take 2-3 minutes)...\n")

# Start server with full output capture
def run_server():
    try:
        result = subprocess.run(
            ["python", "Evo1_server.py"],
            capture_output=True,
            text=True,
            cwd="/content/Evo-1/Evo_1/scripts"
        )
        print("\n=== SERVER OUTPUT ===")
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print("\n=== SERVER ERRORS ===")
            print(result.stderr)
        print(f"\n=== Server exited with code: {result.returncode} ===")
    except Exception as e:
        print(f"\n=== Server exception: {e} ===")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for server to fully load
print("Waiting for server to load model...")
time.sleep(120)  # 2 minutes for model loading

print("\n✅ Server should be ready on ws://localhost:9000")
print("Check output above for any errors")
print("\nIf you see errors, scroll up to see the full server log")
print("Proceeding to client setup...")

In [ ]:
import subprocess
import threading
import time

# Update libero_client_4tasks.py to use localhost
client_file = "/content/LIBERO_evaluation/LIBERO/libero_client_4tasks.py"

# If you don't have the client file yet, upload it first or create it here
if os.path.exists(client_file):
    with open(client_file, 'r') as f:
        client_content = f.read()
    
    # Update SERVER_URL to localhost
    import re
    client_content = re.sub(
        r'SERVER_URL = "[^"]*"',
        'SERVER_URL = "ws://localhost:9000"',
        client_content
    )
    
    with open(client_file, 'w') as f:
        f.write(client_content)
    
    print("✅ Client configured to use ws://localhost:9000")
else:
    print("⚠️  Client file not found!")
    print("Please upload libero_client_4tasks.py to /content/LIBERO_evaluation/LIBERO/")
    print("Then re-run this cell.")

print("\n🚀 Starting LIBERO client in conda environment...")
print("This will run the full evaluation (2-3 hours)...\n")

# Run client in background thread using conda
def run_client():
    os.chdir("/content/LIBERO_evaluation/LIBERO")
    result = subprocess.run(
        ["conda", "run", "-n", "libero", "python", "libero_client_4tasks.py"],
        capture_output=False,
        text=True
    )
    print(f"\nClient finished with code: {result.returncode}")

client_thread = threading.Thread(target=run_client, daemon=False)
client_thread.start()

print("✅ Both server and client are now running!")
print("\n📊 Monitor progress:")
print("   - Server logs will appear above")
print("   - Client logs will stream below")
print("   - This will take 2-3 hours for full evaluation")
print("\n⚠️  Keep this notebook running until completion!")